In [ ]:
# ============================================================
# ENVIRONMENT DETECTION
# ============================================================
import os
import sys

IS_KAGGLE = os.path.exists('/kaggle/input') 
IS_LOCAL  = not IS_KAGGLE

if IS_KAGGLE:
    DATA_DIR   = '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1'   # ← update with actual dataset name
    IMAGE_DIR  = os.path.join(DATA_DIR, 'images')
    TRAIN_CSV  = os.path.join(DATA_DIR, 'train.csv')
    TEST_CSV   = os.path.join(DATA_DIR, 'test.csv')
    SAMPLE_CSV = os.path.join(DATA_DIR, 'sample_submission.csv')
    OUTPUT_DIR = '/kaggle/working'
else:
    DATA_DIR   = '/Users/sanskar/dev/NPPE1'
    IMAGE_DIR  = os.path.join(DATA_DIR, 'images')
    TRAIN_CSV  = os.path.join(DATA_DIR, 'train.csv')
    TEST_CSV   = os.path.join(DATA_DIR, 'test.csv')
    SAMPLE_CSV = os.path.join(DATA_DIR, 'sample_submission.csv')
    OUTPUT_DIR = DATA_DIR

print(f'Environment: {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Data dir: {DATA_DIR}')

In [ ]:
# ============================================================
# INSTALL / IMPORT DEPENDENCIES
# ============================================================
# On Kaggle, most are pre-installed. timm may need install.
if IS_KAGGLE:
    os.system('pip install timm -q')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import random
import warnings
warnings.filterwarnings('ignore')

# Deep learning (Kaggle only — skip locally if not installed)
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
    # Use new torch.amp API (PyTorch 2.0+); fall back for older builds
    try:
        from torch.amp import autocast, GradScaler
    except ImportError:
        from torch.cuda.amp import autocast, GradScaler
    import torchvision.transforms as T
    import timm
    from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, OneCycleLR
    TORCH_AVAILABLE = True
    print(f'PyTorch {torch.__version__}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
    elif torch.backends.mps.is_available():
        print('Device: Apple MPS')
    else:
        print('Device: CPU only')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch not available (EDA-only mode)')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

print('✓ Imports done')

In [ ]:
# ============================================================
# HYPERPARAMETERS & CONFIG
# ============================================================
CFG = {
    # Data
    'img_size'       : 384,           # Native resolution
    'train_img_size' : 320,           # Resize for training (speed/memory tradeoff)
    'val_img_size'   : 384,           # Full resolution for validation/inference
    'num_classes'    : 20,
    'seed'           : 42,
    
    # Model
    'backbone'       : 'tf_efficientnet_b4_ns',  # Noisy Student EfficientNet-B4
    'pretrained'     : True,
    'dropout'        : 0.3,
    
    # Training
    'epochs'         : 30,
    'batch_size'     : 32,            # Adjust for GPU memory
    'val_batch_size' : 64,
    'num_workers'    : 4,
    'lr'             : 3e-4,
    'min_lr'         : 1e-6,
    'weight_decay'   : 1e-4,
    'label_smoothing': 0.1,
    
    # Loss
    'loss_fn'        : 'focal',       # 'ce', 'focal', 'asymmetric'
    'focal_gamma'    : 2.0,
    'use_class_weights': True,
    
    # Augmentation
    'mixup_alpha'    : 0.2,
    'use_tta'        : True,
    'tta_n'          : 4,
    
    # Scoring
    'fn_penalty'     : 5,             # Competition: FN costs 5x more than FP
    
    # Val split
    'val_fold'       : 0,
    'n_folds'        : 5,
}

CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion',
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass',
    'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax',
    'Pneumoperitoneum', 'Pneumomediastinum', 'Subcutaneous Emphysema',
    'Tortuous Aorta', 'Calcification of the Aorta', 'No Finding'
]

# Device
if TORCH_AVAILABLE:
    if torch.cuda.is_available():
        DEVICE = torch.device('cuda')
    elif torch.backends.mps.is_available():
        DEVICE = torch.device('mps')
    else:
        DEVICE = torch.device('cpu')
    print(f'Using device: {DEVICE}')

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
sample_df = pd.read_csv(SAMPLE_CSV)

print(f'Train: {train_df.shape}  |  Test: {test_df.shape}')
print(f'Classes: {len(CLASSES)}')
print()
print('Train head:')
train_df.head(3)

In [ ]:
# ============================================================
# LABEL STRUCTURE VERIFICATION
# ============================================================
row_sums = train_df[CLASSES].sum(axis=1)

print('=== Label Distribution per Row ===')
print(row_sums.value_counts().to_string())
print()
print(f'Single-label (sum==1): {(row_sums==1).sum()} / {len(train_df)}  ← Pure single-label task!')
print(f'Multi-label  (sum>1) : {(row_sums>1).sum()}')
print(f'Unlabeled    (sum==0): {(row_sums==0).sum()}')

In [ ]:
# ============================================================
# CLASS DISTRIBUTION
# ============================================================
class_counts = train_df[CLASSES].sum().sort_values(ascending=False)
class_pcts   = class_counts / len(train_df) * 100

print('Class Distribution:')
print(f'{"Class":<35} {"Count":>8}  {"Pct":>8}')
print('-' * 55)
for cls in class_counts.index:
    print(f'{cls:<35} {int(class_counts[cls]):>8}  {class_pcts[cls]:>7.2f}%')

print()
print(f'Imbalance ratio (max/min): {class_counts.max()/class_counts.min():.1f}x')
print(f'"No Finding" dominates: {class_pcts["No Finding"]:.1f}% of data')

In [ ]:
# ============================================================
# CLASS DISTRIBUTION — VISUALIZATION
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Bar chart
colors = ['#e74c3c' if cls == 'No Finding' else '#3498db' 
          for cls in class_counts.index]
ax = axes[0]
bars = ax.barh(range(len(class_counts)), class_counts.values, color=colors)
ax.set_yticks(range(len(class_counts)))
ax.set_yticklabels(class_counts.index, fontsize=9)
ax.set_xlabel('Count')
ax.set_title('Class Distribution (Linear Scale)', fontsize=12, fontweight='bold')
for i, (cls, cnt) in enumerate(class_counts.items()):
    ax.text(cnt + 50, i, f'{cnt:,}', va='center', fontsize=7.5)
ax.invert_yaxis()

# Log scale
ax2 = axes[1]
ax2.barh(range(len(class_counts)), class_counts.values, color=colors)
ax2.set_yticks(range(len(class_counts)))
ax2.set_yticklabels(class_counts.index, fontsize=9)
ax2.set_xlabel('Count (log scale)')
ax2.set_xscale('log')
ax2.set_title('Class Distribution (Log Scale) — Reveals Extreme Imbalance', 
              fontsize=12, fontweight='bold')
ax2.invert_yaxis()
ax2.axvline(x=class_counts.min(), color='orange', linestyle='--', alpha=0.7, label=f'Min: {class_counts.min()}')
ax2.axvline(x=class_counts.max(), color='red', linestyle='--', alpha=0.7, label=f'Max: {class_counts.max()}')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to class_distribution.png')

In [ ]:
# ============================================================
# SAMPLE IMAGES PER CLASS
# ============================================================
n_cols = 5
n_rows = 4  # show 5 classes × 4 samples

fig = plt.figure(figsize=(20, 20))
# Pick 4 classes to visualize in detail
viz_classes = ['No Finding', 'Infiltration', 'Atelectasis', 'Effusion',
               'Nodule', 'Cardiomegaly', 'Pneumothorax', 'Consolidation']

for row_idx, cls in enumerate(viz_classes):
    subset = train_df[train_df[cls] == 1]['id'].tolist()
    samples = random.sample(subset, min(n_cols, len(subset)))
    
    for col_idx, fn in enumerate(samples):
        ax = fig.add_subplot(len(viz_classes), n_cols, row_idx * n_cols + col_idx + 1)
        img_path = os.path.join(IMAGE_DIR, fn)
        if os.path.exists(img_path):
            img = Image.open(img_path).convert('L')
            ax.imshow(img, cmap='gray')
        ax.axis('off')
        if col_idx == 0:
            ax.set_title(f'{cls}\n(n={len(subset)})', 
                        fontsize=9, fontweight='bold', color='navy', ha='left')

plt.suptitle('Sample Chest X-Rays per Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_images.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# IMAGE PROPERTIES ANALYSIS
# ============================================================
sample_ids = train_df['id'].sample(300, random_state=42).tolist()

sizes, modes, means_r, stds_r = [], [], [], []
for fn in sample_ids:
    fpath = os.path.join(IMAGE_DIR, fn)
    if not os.path.exists(fpath):
        continue
    img = Image.open(fpath)
    sizes.append(img.size)
    modes.append(img.mode)
    arr = np.array(img.convert('RGB')).astype(np.float32) / 255.0
    means_r.append(arr[..., 0].mean())  # R=G=B for grayscale-in-RGB
    stds_r.append(arr[..., 0].std())

print('=== Image Properties ===')
print(f'Unique sizes : {set(sizes)}')
print(f'Unique modes : {set(modes)}')
print(f'Global mean  : {np.mean(means_r):.4f}  ← close to 0.5')
print(f'Global std   : {np.mean(stds_r):.4f}  ← moderate contrast')
print(f'Note: RGB channels are identical (grayscale stored as RGB)')
print()

# Distribution of per-image means
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(means_r, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Per-Image Mean Intensity')
axes[0].set_xlabel('Mean Pixel Value (0-1)')
axes[0].axvline(np.mean(means_r), color='red', linestyle='--', label=f'Mean={np.mean(means_r):.3f}')
axes[0].legend()

axes[1].hist(stds_r, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Per-Image Std Intensity')
axes[1].set_xlabel('Std Pixel Value (0-1)')
axes[1].axvline(np.mean(stds_r), color='red', linestyle='--', label=f'Mean={np.mean(stds_r):.3f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'image_stats.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('→ Images are well-exposed with symmetric distribution around 0.5')
print('→ Use mean=0.5, std=0.23 for normalization (or ImageNet stats for pretrained models)')

In [ ]:
# ============================================================
# PER-CLASS IMAGE STATISTICS
# ============================================================
print('=== Per-Class Mean Pixel Intensity (10 samples each) ===')
random.seed(42)

cls_means, cls_stds = {}, {}
for cls in CLASSES:
    subset = train_df[train_df[cls] == 1]['id'].tolist()
    n_sample = min(10, len(subset))
    sample = random.sample(subset, n_sample)
    
    m_list, s_list = [], []
    for fn in sample:
        fpath = os.path.join(IMAGE_DIR, fn)
        if os.path.exists(fpath):
            arr = np.array(Image.open(fpath).convert('L')).astype(np.float32)
            m_list.append(arr.mean())
            s_list.append(arr.std())
    cls_means[cls] = np.mean(m_list)
    cls_stds[cls]  = np.mean(s_list)

cls_means_s = pd.Series(cls_means).sort_values()
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c' if cls == 'No Finding' else '#2980b9' for cls in cls_means_s.index]
ax.barh(range(len(cls_means_s)), cls_means_s.values, color=colors)
ax.set_yticks(range(len(cls_means_s)))
ax.set_yticklabels(cls_means_s.index, fontsize=9)
ax.set_xlabel('Mean Pixel Intensity (0-255)')
ax.set_title('Mean Pixel Intensity per Class', fontweight='bold')
ax.axvline(x=np.mean(list(cls_means.values())), color='orange', linestyle='--', 
           label=f'Overall mean: {np.mean(list(cls_means.values())):.1f}')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'per_class_intensity.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# ASYMMETRIC SCORING FUNCTION
# ============================================================
def competition_score(y_true: np.ndarray, y_pred: np.ndarray, 
                      classes=CLASSES, verbose=True) -> float:
    """
    Compute the macro-averaged asymmetric competition score.
    
    Args:
        y_true: (N,) array of true class indices
        y_pred: (N,) array of predicted class indices
        classes: list of class names
        verbose: print per-class scores
    Returns:
        macro-averaged score
    """
    C = len(classes)
    scores = []
    
    for c_idx, cls in enumerate(classes):
        N_c = (y_true == c_idx).sum()
        if N_c == 0:
            continue  # Skip classes not in ground truth
        
        TP = ((y_true == c_idx) & (y_pred == c_idx)).sum()
        FP = ((y_true != c_idx) & (y_pred == c_idx)).sum()
        FN = ((y_true == c_idx) & (y_pred != c_idx)).sum()
        
        score_c = (TP - FP - 5 * FN) / N_c
        scores.append(score_c)
        
        if verbose:
            print(f'{cls:<35} N={N_c:5d} | TP={TP:5d} FP={FP:5d} FN={FN:5d} | Score={score_c:7.4f}')
    
    macro_avg = np.mean(scores)
    if verbose:
        print('-' * 75)
        print(f'{"MACRO AVERAGE":<35}                               | Score={macro_avg:7.4f}')
    return macro_avg


def optimal_predict(probs: np.ndarray, class_counts: np.ndarray) -> np.ndarray:
    """
    Bayes-optimal prediction: argmax_c [(7*P(c|x) - 1) / N_c]
    
    Args:
        probs: (N, C) softmax probabilities
        class_counts: (C,) number of training samples per class
    Returns:
        (N,) predicted class indices
    """
    # Score for each class: (7*p - 1) / N_c
    decision_scores = (7 * probs - 1) / class_counts[np.newaxis, :]  # (N, C)
    return np.argmax(decision_scores, axis=1)


print('Scoring functions defined.')
print()
# Verify on a trivial example
y_true_ex = np.array([0, 0, 1, 1, 2])  # 3 classes
y_pred_ex = np.array([0, 1, 1, 1, 2])  # perfect except one FP
print('Trivial example (3 classes, near-perfect):')
_ = competition_score(y_true_ex, y_pred_ex, classes=['A','B','C'])

In [ ]:
# ============================================================
# BASELINE SCORE ANALYSIS
# ============================================================
class_counts_arr = np.array([class_counts[cls] for cls in CLASSES])
class_to_idx = {cls: i for i, cls in enumerate(CLASSES)}

# Get integer labels for train
y_train = train_df[CLASSES].values.argmax(axis=1)

print('=== Baseline Strategy Analysis ===')
print()

# 1. Always predict No Finding
nf_idx = class_to_idx['No Finding']
y_always_nf = np.full(len(y_train), nf_idx)
print('Strategy 1: Always predict "No Finding"')
score_nf = competition_score(y_train, y_always_nf, verbose=False)
print(f'  Score: {score_nf:.4f}  (terrible — all disease FNs)')
print()

# 2. Perfect predictions
print('Strategy 2: Perfect predictions')
score_perfect = competition_score(y_train, y_train, verbose=False)
print(f'  Score: {score_perfect:.4f}  (upper bound = 1.0)')
print()

# 3. Random predictions
np.random.seed(42)
y_random = np.random.randint(0, len(CLASSES), len(y_train))
print('Strategy 3: Completely random')
score_random = competition_score(y_train, y_random, verbose=False)
print(f'  Score: {score_random:.4f}')
print()

# 4. Frequency-based (predict by class frequency)
y_freq = np.random.choice(len(CLASSES), len(y_train), 
                          p=class_counts_arr/class_counts_arr.sum())
print('Strategy 4: Random with class frequency weights')
score_freq = competition_score(y_train, y_freq, verbose=False)
print(f'  Score: {score_freq:.4f}')

In [ ]:
# ============================================================
# VISUALIZE SCORING STRUCTURE
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: What probability of rare disease needed to beat P(No Finding)=0.5
N_NF = class_counts['No Finding']
nf_50_score = (7 * 0.5 - 1) / N_NF

min_probs = {}
for cls in CLASSES:
    if cls == 'No Finding':
        continue
    N_c = class_counts[cls]
    min_p = (nf_50_score * N_c + 1) / 7
    min_probs[cls] = min_p

mp_series = pd.Series(min_probs).sort_values(ascending=False)
ax = axes[0]
ax.barh(range(len(mp_series)), mp_series.values, color='#e74c3c', alpha=0.8)
ax.set_yticks(range(len(mp_series)))
ax.set_yticklabels(mp_series.index, fontsize=9)
ax.set_xlabel('Minimum P(class) to beat P(No Finding)=0.5')
ax.set_title('Optimal Decision Thresholds', fontweight='bold')
ax.axvline(x=1/7, color='blue', linestyle='--', label=f'1/7 ≈ 0.143 (absolute min)')
ax.legend()
ax.invert_yaxis()

# Panel 2: Score sensitivity to FN vs FP tradeoff
ax2 = axes[1]
n_samples = 1000
fn_rates = np.linspace(0, 0.5, 50)
fp_rates = np.linspace(0, 0.5, 50)

# For a single class with N=1000
for fn_frac in [0.0, 0.1, 0.2]:
    scores = []
    for fp_frac in fp_rates:
        TP = int(n_samples * (1 - fn_frac))
        FN = n_samples - TP
        FP = int(n_samples * fp_frac)
        score = (TP - FP - 5 * FN) / n_samples
        scores.append(score)
    ax2.plot(fp_rates, scores, label=f'FN rate={fn_frac}')

ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('Score_c')
ax2.set_title('Score Sensitivity: FP Rate vs Score\n(for fixed FN rates)', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'scoring_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: Even a small FN rate dominates the score. Keep FN rate LOW.')

In [ ]:
# ============================================================
# CLASS WEIGHT COMPUTATION
# ============================================================
# Method 1: Inverse frequency weights
total_samples = len(train_df)
inv_freq_weights = total_samples / (len(CLASSES) * class_counts_arr)

# Method 2: Effective Number of Samples (Cui et al., 2019)
# β = (N-1)/N where N = class count; weight = (1-β)/(1-β^N_c)
beta = 0.9999
en_weights = (1 - beta) / (1 - beta ** class_counts_arr)
en_weights_norm = en_weights / en_weights.sum() * len(CLASSES)

# Method 3: Score-aware weights (upweight rare classes by 1/N_c factor)
# The competition score is 1/N_c weighted, so naturally incorporate this
score_aware_weights = 1.0 / class_counts_arr
score_aware_weights = score_aware_weights / score_aware_weights.sum() * len(CLASSES)

print('Class Weights Comparison:')
print(f'{"Class":<35} {"InvFreq":>10} {"EffNum":>10} {"ScoreAware":>12}')
print('-' * 70)
for i, cls in enumerate(CLASSES):
    print(f'{cls:<35} {inv_freq_weights[i]:>10.3f} {en_weights_norm[i]:>10.3f} {score_aware_weights[i]:>12.4f}')

# Use inverse frequency for cross-entropy (clipped to avoid extremes)
CLASS_WEIGHTS = np.clip(inv_freq_weights, a_min=None, a_max=10.0)  # Cap at 10x
CLASS_WEIGHTS = CLASS_WEIGHTS / CLASS_WEIGHTS.sum() * len(CLASSES)  # Re-normalize
print()
print('Final class weights (inv_freq, clipped at 50x, renormalized):')
for i, cls in enumerate(CLASSES):
    print(f'  {cls}: {CLASS_WEIGHTS[i]:.4f}')

In [ ]:
# ============================================================
# TRAIN / VALIDATION SPLIT (Stratified)
# ============================================================
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])

train_df = train_df.reset_index(drop=True)
y_labels = train_df[CLASSES].values.argmax(axis=1)

train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, y_labels)):
    train_df.loc[val_idx, 'fold'] = fold

# Add integer label column
train_df['label'] = y_labels

fold = CFG['val_fold']
df_trn = train_df[train_df['fold'] != fold].reset_index(drop=True)
df_val = train_df[train_df['fold'] == fold].reset_index(drop=True)

print(f'Train fold: {len(df_trn):,} samples')
print(f'Val fold:   {len(df_val):,} samples')
print()
# Verify stratification
for cls in ['No Finding', 'Infiltration', 'Pneumomediastinum']:
    c_idx = class_to_idx[cls]
    trn_pct = (df_trn['label'] == c_idx).mean() * 100
    val_pct = (df_val['label'] == c_idx).mean() * 100
    print(f'{cls}: train={trn_pct:.2f}%, val={val_pct:.2f}%')

In [ ]:
# ============================================================
# TRANSFORMS
# ============================================================
if TORCH_AVAILABLE:
    # ImageNet stats work well for pretrained models
    # (dataset mean ≈ 0.502 ≈ ImageNet mean 0.485 for grayscale-as-RGB)
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]
    
    # Dataset-specific stats (computed above)
    DATASET_MEAN = [0.502, 0.502, 0.502]
    DATASET_STD  = [0.231, 0.231, 0.231]
    
    NORM_MEAN = IMAGENET_MEAN  # Use ImageNet for pretrained
    NORM_STD  = IMAGENET_STD
    
    train_transform = T.Compose([
        T.Resize((CFG['train_img_size'], CFG['train_img_size'])),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        T.RandomAffine(
            degrees=0,
            translate=(0.1, 0.1),
            scale=(0.85, 1.15),
            shear=10,
        ),
        T.ColorJitter(
            brightness=0.3,
            contrast=0.3,
        ),
        T.ToTensor(),
        T.Normalize(mean=NORM_MEAN, std=NORM_STD),
        T.RandomErasing(p=0.2, scale=(0.02, 0.1)),  # Must be after ToTensor (needs tensor input)
    ])
    
    val_transform = T.Compose([
        T.Resize((CFG['val_img_size'], CFG['val_img_size'])),
        T.ToTensor(),
        T.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ])
    
    # TTA transforms
    tta_transforms = [
        val_transform,  # Original
        T.Compose([T.Resize((CFG['val_img_size'], CFG['val_img_size'])),
                   T.RandomHorizontalFlip(p=1.0),
                   T.ToTensor(), T.Normalize(NORM_MEAN, NORM_STD)]),
        T.Compose([T.Resize((int(CFG['val_img_size']*1.1), int(CFG['val_img_size']*1.1))),
                   T.CenterCrop(CFG['val_img_size']),
                   T.ToTensor(), T.Normalize(NORM_MEAN, NORM_STD)]),
        T.Compose([T.Resize((CFG['val_img_size'], CFG['val_img_size'])),
                   T.RandomRotation(degrees=(10, 10)),
                   T.ToTensor(), T.Normalize(NORM_MEAN, NORM_STD)]),
    ]
    
    print('Transforms defined:')
    print(f'  Train: {CFG["train_img_size"]}x{CFG["train_img_size"]} + augmentation')
    print(f'  Val:   {CFG["val_img_size"]}x{CFG["val_img_size"]} + no augmentation')
    print(f'  TTA:   {len(tta_transforms)} transforms')

In [ ]:
# ============================================================
# DATASET CLASS
# ============================================================
if TORCH_AVAILABLE:
    class ChestXRayDataset(Dataset):
        def __init__(self, df, image_dir, transform=None, mode='train'):
            """
            Args:
                df: DataFrame with 'id' and 'label' columns
                image_dir: path to images folder
                transform: torchvision transform
                mode: 'train', 'val', or 'test'
            """
            self.df = df.reset_index(drop=True)
            self.image_dir = image_dir
            self.transform = transform
            self.mode = mode
        
        def __len__(self):
            return len(self.df)
        
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img_path = os.path.join(self.image_dir, row['id'])
            
            img = Image.open(img_path).convert('RGB')
            
            if self.transform:
                img = self.transform(img)
            
            if self.mode == 'test':
                return img, row['id']
            else:
                label = int(row['label'])
                return img, label
    
    print('ChestXRayDataset class defined.')
    
    # Test dataset instantiation
    test_ds = ChestXRayDataset(df_trn.head(4), IMAGE_DIR, val_transform, mode='train')
    sample_img, sample_lbl = test_ds[0]
    print(f'Sample image tensor shape: {sample_img.shape}')
    print(f'Sample label: {sample_lbl} → {CLASSES[sample_lbl]}')

In [ ]:
# ============================================================  
# DATALOADERS — natural sampling + class-weighted loss                                                                                                                                                                                                                             
# ============================================================  
# NOTE: WeightedRandomSampler is intentionally NOT used.
# Reason: it trains the model to expect a uniform class distribution,
# causing val accuracy to collapse (~11%) on the real distribution (67% NF).                                                                                                                                                                                                       
# Class imbalance is handled entirely by the loss function class weights.
if TORCH_AVAILABLE:                                                                                                                                                                                                                                                                
  train_ds = ChestXRayDataset(df_trn, IMAGE_DIR, train_transform, mode='train')
  val_ds   = ChestXRayDataset(df_val, IMAGE_DIR, val_transform,   mode='val')                                                                                                                                                                                                    
  test_ds  = ChestXRayDataset(
      test_df.assign(label=0),                                                                                                                                                                                                                                                   
      IMAGE_DIR, val_transform, mode='val'  # mode='val' so validate() gets integer labels
  )                                                                                                                                                                                                                                                                              
                                                              
  _pin = (DEVICE.type == 'cuda')  # pin_memory only speeds up CUDA transfers                                                                                                                                                                                                     

  train_loader = DataLoader(                                                                                                                                                                                                                                                     
      train_ds,                                               
      batch_size=CFG['batch_size'],
      shuffle=True,              # Natural distribution — class weights handle imbalance
      num_workers=CFG['num_workers'],
      pin_memory=_pin,                                                                                                                                                                                                                                                           
      drop_last=True,
  )                                                                                                                                                                                                                                                                              
                                                              
  val_loader = DataLoader(
      val_ds,
      batch_size=CFG['val_batch_size'],
      shuffle=False,
      num_workers=CFG['num_workers'],                                                                                                                                                                                                                                            
      pin_memory=_pin,
  )                                                                                                                                                                                                                                                                              
                                                              
  test_loader = DataLoader(
      test_ds,
      batch_size=CFG['val_batch_size'],
      shuffle=False,
      num_workers=CFG['num_workers'],
      pin_memory=_pin,                                                                                                                                                                                                                                                           
  )
                                                                                                                                                                                                                                                                                 
  print(f'Train loader: {len(train_loader)} batches × {CFG["batch_size"]}')
  print(f'Val loader  : {len(val_loader)} batches × {CFG["val_batch_size"]}')
  print(f'Test loader : {len(test_loader)} batches × {CFG["val_batch_size"]}')                                                                                                                                                                                                   
   

In [ ]:
# ============================================================
# VISUALIZE AUGMENTED SAMPLES
# ============================================================
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(4, 6, figsize=(20, 14))
    
    # Show original vs augmented for 4 samples
    sample_indices = random.sample(range(len(df_trn)), 4)
    
    for row, idx in enumerate(sample_indices):
        row_data = df_trn.iloc[idx]
        img_path = os.path.join(IMAGE_DIR, row_data['id'])
        orig_img = Image.open(img_path).convert('RGB')
        cls_name = CLASSES[int(row_data['label'])]
        
        # Original
        axes[row, 0].imshow(orig_img, cmap='gray')
        axes[row, 0].set_title(f'Original\n{cls_name}', fontsize=8)
        axes[row, 0].axis('off')
        
        # 5 augmented versions
        for aug_idx in range(5):
            aug_tensor = train_transform(orig_img)
            # Denormalize for display
            mean = torch.tensor(NORM_MEAN).view(3, 1, 1)
            std  = torch.tensor(NORM_STD).view(3, 1, 1)
            aug_img = (aug_tensor * std + mean).clamp(0, 1)
            axes[row, aug_idx+1].imshow(aug_img.permute(1,2,0))
            axes[row, aug_idx+1].set_title(f'Aug {aug_idx+1}', fontsize=8)
            axes[row, aug_idx+1].axis('off')
    
    plt.suptitle('Augmentation Examples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'augmentation_examples.png'), dpi=120)
    plt.show()

In [ ]:
# ============================================================
# MODEL DEFINITION
# ============================================================
if TORCH_AVAILABLE:
    class ChestXRayModel(nn.Module):
        def __init__(self, backbone_name, num_classes, pretrained=True, dropout=0.3):
            super().__init__()
            self.backbone = timm.create_model(
                backbone_name,
                pretrained=pretrained,
                num_classes=0,         # Remove classifier head
                global_pool='avg',     # Global average pooling
            )
            in_features = self.backbone.num_features
            
            self.head = nn.Sequential(
                nn.BatchNorm1d(in_features),
                nn.Dropout(dropout),
                nn.Linear(in_features, 512),
                nn.SiLU(),             # Swish activation
                nn.BatchNorm1d(512),
                nn.Dropout(dropout / 2),
                nn.Linear(512, num_classes),
            )
            
            # Initialize classifier head
            nn.init.xavier_uniform_(self.head[-1].weight)
            nn.init.zeros_(self.head[-1].bias)
        
        def forward(self, x):
            features = self.backbone(x)   # (B, in_features)
            logits   = self.head(features) # (B, num_classes)
            return logits
        
        def get_features(self, x):
            """Get intermediate features for analysis."""
            return self.backbone(x)
    
    # Instantiate model
    model = ChestXRayModel(
        backbone_name=CFG['backbone'],
        num_classes=CFG['num_classes'],
        pretrained=CFG['pretrained'],
        dropout=CFG['dropout'],
    )
    model = model.to(DEVICE)
    
    # Model summary
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model: {CFG["backbone"]}')
    print(f'Total params    : {total_params:,}')
    print(f'Trainable params: {trainable_params:,}')
    
    # Test forward pass
    dummy = torch.randn(2, 3, CFG['train_img_size'], CFG['train_img_size']).to(DEVICE)
    with torch.no_grad():
        out = model(dummy)
    print(f'Output shape: {out.shape}  ← (batch, {len(CLASSES)} classes)')

In [ ]:
# ============================================================
# LOSS FUNCTIONS
# ============================================================
if TORCH_AVAILABLE:
    class FocalLoss(nn.Module):
        """
        Focal Loss for multi-class classification.
        FL(pt) = -α_t * (1 - pt)^γ * log(pt)
        """
        def __init__(self, gamma=2.0, weight=None):
            super().__init__()
            self.gamma = gamma
            self.weight = weight

        
        def forward(self, logits, targets):
            log_p = F.log_softmax(logits, dim=1)             # (B, C)
            # Unweighted CE for correct p_t in focal modulation
            ce_unweighted = F.nll_loss(log_p, targets, reduction='none')  # (B,)
            p_t = torch.exp(-ce_unweighted)
            focal_weight = (1 - p_t) ** self.gamma               # (B,)
            # Weighted CE for the actual loss value
            if self.weight is not None:
                ce_weighted = F.nll_loss(
                    log_p, targets,
                    weight=self.weight,
                    reduction='none',
                )                                                 # (B,)
            else:
                ce_weighted = ce_unweighted
            return (focal_weight * ce_weighted).mean()
    
    
    class AsymmetricCompetitionLoss(nn.Module):
        """
        Loss that directly penalizes false negatives 5x more.
        Uses soft labels to make it differentiable.
        
        For class c:
          - If truth=c: maximize P(c|x)  → reward TP
          - If truth≠c: minimize P(c|x)  → penalize FP
        With asymmetric weights (FN weighted 5x vs FP).
        """
        def __init__(self, fn_penalty=5, class_counts=None):
            super().__init__()
            self.fn_penalty = fn_penalty
            self.class_counts = class_counts  # (C,) numpy array
        
        def forward(self, logits, targets):
            B, C = logits.shape
            probs = F.softmax(logits, dim=1)  # (B, C)
            
            # One-hot targets
            one_hot = F.one_hot(targets, num_classes=C).float()  # (B, C)
            
            # TP contribution: p_c when truth=c
            tp_term = (one_hot * probs).sum(dim=1).clamp(min=1e-7).log()  # (B,)
            
            # FN contribution: penalize low p_c when truth=c (complement of TP)
            fn_term = (one_hot * (1 - probs)).sum(dim=1).clamp(min=1e-7).log()  # (B,)
            
            # FP contribution: penalize high p_c when truth≠c
            fp_term = ((1 - one_hot) * probs).sum(dim=1).clamp(min=1e-7).log()  # (B,)
            
            # Asymmetric loss: maximize TP, minimize FN (5x), minimize FP (1x)
            loss = -(tp_term - self.fn_penalty * fn_term - fp_term).mean()
            return loss
    
    
    # Instantiate the chosen loss
    class_weights_tensor = torch.FloatTensor(CLASS_WEIGHTS).to(DEVICE)
    
    if CFG['loss_fn'] == 'ce':
        criterion = nn.CrossEntropyLoss(
            weight=class_weights_tensor,
            label_smoothing=CFG['label_smoothing'],
        )
    elif CFG['loss_fn'] == 'focal':
        criterion = FocalLoss(
            gamma=CFG['focal_gamma'],
            weight=class_weights_tensor,
        )
    elif CFG['loss_fn'] == 'asymmetric':
        criterion = AsymmetricCompetitionLoss(
            fn_penalty=CFG['fn_penalty'],
            class_counts=class_counts_arr,
        )
    
    print(f'Loss function: {CFG["loss_fn"]}')
    
    # Test loss on dummy data
    dummy_logits = torch.randn(4, len(CLASSES)).to(DEVICE)
    dummy_labels = torch.randint(0, len(CLASSES), (4,)).to(DEVICE)
    dummy_loss = criterion(dummy_logits, dummy_labels)
    print(f'Test loss value: {dummy_loss.item():.4f}')

In [ ]:
# 🚀 KAGGLE GPU — OPTIMIZER & SCHEDULER
if TORCH_AVAILABLE:
    # Use differential learning rates:
    # - Backbone: lower lr (already pretrained)
    # - Head: higher lr (randomly initialized)
    backbone_params = [p for p in model.backbone.parameters()]
    head_params     = [p for p in model.head.parameters()]
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': CFG['lr'] * 0.1},
        {'params': head_params,     'lr': CFG['lr']},
    ], weight_decay=CFG['weight_decay'])
    
    # OneCycleLR: warmup + cosine decay in one pass
    scheduler = OneCycleLR(
        optimizer,
        max_lr=[CFG['lr'] * 0.1, CFG['lr']],
        epochs=CFG['epochs'],
        steps_per_epoch=len(train_loader),
        pct_start=0.1,          # 10% warmup
        anneal_strategy='cos',
        div_factor=25,
        final_div_factor=1e4,
    )
    
    # Mixed precision scaler — GradScaler(device) is the new API (PyTorch 2.0+)
    _amp_device = DEVICE.type if DEVICE.type in ('cuda', 'cpu') else 'cpu'
    scaler = GradScaler(_amp_device, enabled=(DEVICE.type == 'cuda'))
    
    print('Optimizer: AdamW with differential learning rates')
    print(f'  Backbone LR: {CFG["lr"]*0.1}')
    print(f'  Head LR    : {CFG["lr"]}')
    print('Scheduler: OneCycleLR')
    print(f'Mixed precision: {DEVICE.type == "cuda"}')

In [ ]:
# 🚀 KAGGLE GPU — MIXUP HELPER
if TORCH_AVAILABLE:
    def mixup_data(x, y, alpha=0.2):
        """Apply Mixup augmentation."""
        if alpha > 0:
            lam = np.random.beta(alpha, alpha)
        else:
            lam = 1.0
        
        batch_size = x.size(0)
        idx = torch.randperm(batch_size, device=x.device)
        
        mixed_x = lam * x + (1 - lam) * x[idx]
        y_a, y_b = y, y[idx]
        return mixed_x, y_a, y_b, lam
    
    def mixup_criterion(criterion, pred, y_a, y_b, lam):
        """Compute Mixup loss."""
        return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)
    
    print('Mixup helpers defined.')

In [ ]:
# 🚀 KAGGLE GPU — TRAINING & VALIDATION FUNCTIONS
if TORCH_AVAILABLE:
    def train_one_epoch(model, loader, optimizer, scheduler, scaler, 
                        criterion, epoch, device):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        
        for batch_idx, (images, labels) in enumerate(loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            # Mixup
            if CFG['mixup_alpha'] > 0:
                images, y_a, y_b, lam = mixup_data(images, labels, CFG['mixup_alpha'])
            
            optimizer.zero_grad()
            
            _amp_dev = device.type if device.type in ('cuda', 'cpu') else 'cpu'
            with autocast(_amp_dev, enabled=(device.type == 'cuda')):
                logits = model(images)
                if CFG['mixup_alpha'] > 0:
                    loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
                else:
                    loss = criterion(logits, labels)
            
            scaler.scale(loss).backward()
            
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            total_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)
            
            if batch_idx % 100 == 0:
                lr_now = scheduler.get_last_lr()
                print(f'  Epoch {epoch} [{batch_idx:4d}/{len(loader)}] '
                      f'Loss: {total_loss/total:.4f} '
                      f'Acc: {100*correct/total:.2f}% '
                      f'LR: {lr_now[-1]:.2e}')
        
        return total_loss / total, correct / total
    
    
    def validate(model, loader, criterion, device):
        model.eval()
        total_loss = 0.0
        all_probs  = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                
                _amp_dev = device.type if device.type in ('cuda', 'cpu') else 'cpu'
                with autocast(_amp_dev, enabled=(device.type == 'cuda')):
                    logits = model(images)
                    loss = criterion(logits, labels)
                
                total_loss += loss.item() * images.size(0)
                probs = F.softmax(logits, dim=1)
                all_probs.append(probs.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        
        all_probs  = np.concatenate(all_probs,  axis=0)  # (N, C)
        all_labels = np.concatenate(all_labels, axis=0)  # (N,)
        avg_loss   = total_loss / len(loader.dataset)
        
        return avg_loss, all_probs, all_labels
    
    print('Training functions defined.')

In [ ]:
# 🚀 KAGGLE GPU — MAIN TRAINING LOOP
if TORCH_AVAILABLE:
    best_score = -float('inf')
    best_model_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
    history = {'train_loss': [], 'val_loss': [], 'val_score': [], 'lr': []}
    
    print(f'Starting training for {CFG["epochs"]} epochs...')
    print('=' * 80)
    
    for epoch in range(1, CFG['epochs'] + 1):
        print(f'\nEpoch {epoch}/{CFG["epochs"]}')
        print('-' * 50)
        
        # Train
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, scaler,
            criterion, epoch, DEVICE
        )
        
        # Validate
        val_loss, val_probs, val_labels = validate(model, val_loader, criterion, DEVICE)
        
        # Compute competition score with optimal decision rule
        val_preds_argmax  = val_probs.argmax(axis=1)  # Standard argmax
        val_preds_optimal = optimal_predict(val_probs, class_counts_arr)  # Optimal rule
        
        score_argmax  = competition_score(val_labels, val_preds_argmax,  verbose=False)
        score_optimal = competition_score(val_labels, val_preds_optimal, verbose=False)
        
        val_acc = (val_preds_argmax == val_labels).mean()
        
        print(f'\n  Train Loss : {train_loss:.4f} | Train Acc : {train_acc*100:.2f}%')
        print(f'  Val Loss   : {val_loss:.4f} | Val Acc   : {val_acc*100:.2f}%')
        print(f'  Val Score (argmax)  : {score_argmax:.4f}')
        print(f'  Val Score (optimal) : {score_optimal:.4f}  ← Competition metric')
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_score'].append(score_optimal)
        history['lr'].append(scheduler.get_last_lr()[-1])
        
        # Save best model
        if score_optimal > best_score:
            best_score = score_optimal
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'score': best_score,
                'class_counts': class_counts_arr,
                'cfg': CFG,
            }, best_model_path)
            print(f'  ★ New best! Score={best_score:.4f} → saved to {best_model_path}')
    
    print(f'\nTraining complete. Best score: {best_score:.4f}')

In [ ]:
# 🚀 KAGGLE GPU — TRAINING CURVES
if TORCH_AVAILABLE and len(history['train_loss']) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train')
    axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val')
    axes[0].set_title('Loss Curve', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs_range, history['val_score'], 'g-o', label='Val Competition Score')
    axes[1].axhline(y=best_score, color='red', linestyle='--', label=f'Best: {best_score:.4f}')
    axes[1].set_title('Competition Score (Validation)', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(epochs_range, history['lr'], 'purple', label='LR')
    axes[2].set_title('Learning Rate Schedule', fontweight='bold')
    axes[2].set_xlabel('Epoch')
    axes[2].set_yscale('log')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
    plt.show()

In [ ]:
# 🚀 KAGGLE GPU — LOAD BEST MODEL FOR ANALYSIS
if TORCH_AVAILABLE:
    checkpoint = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'Loaded best model from epoch {checkpoint["epoch"]}')
    print(f'Best validation score: {checkpoint["score"]:.4f}')
    
    # Get full validation predictions
    _, val_probs, val_labels = validate(model, val_loader, criterion, DEVICE)
    print(f'Validation set: {len(val_labels):,} samples')

In [ ]:
# 🚀 KAGGLE GPU — DETAILED VALIDATION ANALYSIS
if TORCH_AVAILABLE:
    # Compute scores for both decision strategies
    val_preds_argmax  = val_probs.argmax(axis=1)
    val_preds_optimal = optimal_predict(val_probs, class_counts_arr)
    
    print('=== Argmax Decision Rule ===')
    score_argmax = competition_score(val_labels, val_preds_argmax)
    print()
    print('=== Optimal Decision Rule (Bayes-optimal for this scoring) ===')
    score_optimal = competition_score(val_labels, val_preds_optimal)

In [ ]:
# 🚀 KAGGLE GPU — CONFUSION MATRIX
if TORCH_AVAILABLE:
    from sklearn.metrics import confusion_matrix
    
    cm = confusion_matrix(val_labels, val_preds_optimal, labels=range(len(CLASSES)))
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, ax = plt.subplots(figsize=(16, 14))
    short_names = [
        'Atl', 'Card', 'Cons', 'Edema', 'Eff', 'Emph', 'Fib', 'Hern',
        'Infil', 'Mass', 'Nod', 'PlThk', 'Pneu', 'PnTx', 'PnPe',
        'PnMe', 'SubE', 'TortA', 'CalcA', 'NoFind'
    ]
    
    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=short_names, yticklabels=short_names,
        ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8}
    )
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_title('Normalized Confusion Matrix (Optimal Decision Rule)', 
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
    plt.show()

In [ ]:
# 🚀 KAGGLE GPU — PER-CLASS PROBABILITY CALIBRATION
if TORCH_AVAILABLE:
    # Check calibration: are predicted probabilities well-calibrated?
    fig, axes = plt.subplots(4, 5, figsize=(20, 16))
    axes = axes.flatten()
    
    for c_idx, cls in enumerate(CLASSES):
        ax = axes[c_idx]
        mask_pos = (val_labels == c_idx)
        mask_neg = (val_labels != c_idx)
        
        if mask_pos.sum() > 0:
            ax.hist(val_probs[mask_pos, c_idx], bins=30, alpha=0.7, 
                   color='green', label=f'True ({mask_pos.sum()})', density=True)
        if mask_neg.sum() > 0:
            ax.hist(val_probs[mask_neg, c_idx], bins=30, alpha=0.7,
                   color='red', label=f'False ({mask_neg.sum()})', density=True)
        
        ax.set_title(f'{cls}\n(N={mask_pos.sum()})', fontsize=8)
        ax.set_xlabel('P(class)', fontsize=7)
        ax.legend(fontsize=6)
        ax.axvline(x=1/7, color='blue', linestyle='--', alpha=0.5, label='1/7')
    
    plt.suptitle('Per-Class Probability Distributions (Green=True, Red=False)',
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'probability_calibration.png'), dpi=120)
    plt.show()

In [ ]:
# 🚀 KAGGLE GPU — INFERENCE WITH TTA
if TORCH_AVAILABLE:
    def predict_with_tta(model, image_ids, image_dir, tta_transforms, 
                         batch_size, device, num_workers=4):
        """
        Run inference with Test Time Augmentation.
        Returns average probabilities across all TTA transforms.
        """
        model.eval()
        all_avg_probs = None
        
        for t_idx, transform in enumerate(tta_transforms):
            print(f'  TTA {t_idx+1}/{len(tta_transforms)}...')
            
            tta_df = pd.DataFrame({'id': image_ids, 'label': 0})
            tta_ds = ChestXRayDataset(tta_df, image_dir, transform, mode='test')
            tta_loader = DataLoader(
                tta_ds, batch_size=batch_size, shuffle=False,
                num_workers=num_workers, pin_memory=(device.type=='cuda')
            )
            
            probs_list = []
            with torch.no_grad():
                for images, _ in tta_loader:
                    images = images.to(device, non_blocking=True)
                    _amp_dev = device.type if device.type in ('cuda', 'cpu') else 'cpu'
                    with autocast(_amp_dev, enabled=(device.type=='cuda')):
                        logits = model(images)
                    probs = F.softmax(logits, dim=1)
                    probs_list.append(probs.cpu().numpy())
            
            probs_arr = np.concatenate(probs_list, axis=0)  # (N, C)
            
            if all_avg_probs is None:
                all_avg_probs = probs_arr
            else:
                all_avg_probs += probs_arr
        
        return all_avg_probs / len(tta_transforms)
    
    
    print('Predicting on test set with TTA...')
    test_image_ids = test_df['id'].tolist()
    
    if CFG['use_tta']:
        test_probs = predict_with_tta(
            model, test_image_ids, IMAGE_DIR,
            tta_transforms[:CFG['tta_n']],
            batch_size=CFG['val_batch_size'],
            device=DEVICE,
            num_workers=CFG['num_workers'],
        )
    else:
        _, test_probs, _ = validate(model, test_loader, criterion, DEVICE)
    
    print(f'Test predictions shape: {test_probs.shape}')
    print(f'Probabilities sum to 1: {test_probs.sum(axis=1).mean():.4f} (should be ~1.0)')

In [ ]:
# 🚀 KAGGLE GPU — APPLY OPTIMAL DECISION RULE & BUILD SUBMISSION
if TORCH_AVAILABLE:
    # Apply the Bayes-optimal decision rule
    test_preds = optimal_predict(test_probs, class_counts_arr)
    
    print('Test prediction distribution:')
    pred_counts = pd.Series(test_preds).map(lambda i: CLASSES[i]).value_counts()
    print(pred_counts.to_string())
    print()
    
    # Build submission dataframe
    submission = sample_df.copy()
    
    # Reset all to 0
    for cls in CLASSES:
        submission[cls] = 0
    
    # Set predicted class to 1
    id_to_idx = {row['id']: i for i, row in submission.iterrows()}
    for img_id, pred_idx in zip(test_image_ids, test_preds):
        if img_id in id_to_idx:
            row_idx = id_to_idx[img_id]
            submission.loc[row_idx, CLASSES[pred_idx]] = 1
    
    # Verify format
    assert (submission[CLASSES].sum(axis=1) == 1).all(), 'Every row should have exactly 1 label!'
    
    submission_path = os.path.join(OUTPUT_DIR, 'submission.csv')
    submission.to_csv(submission_path, index=False)
    
    print(f'Submission saved to: {submission_path}')
    print(f'Shape: {submission.shape}')
    print()
    print('First few rows:')
    submission.head()

In [ ]:
# 🚀 KAGGLE GPU — SUBMISSION SANITY CHECK
if TORCH_AVAILABLE:
    print('=== Submission Sanity Check ===')
    print(f'Total rows: {len(submission)}')
    print(f'Expected  : {len(test_df)}')
    print()
    
    # Check all IDs present
    expected_ids = set(test_df['id'].tolist())
    submission_ids = set(submission['id'].tolist())
    print(f'All test IDs present: {expected_ids == submission_ids}')
    print(f'Missing IDs: {len(expected_ids - submission_ids)}')
    print()
    
    # Check label counts
    row_sums = submission[CLASSES].sum(axis=1)
    print(f'Single-label rows: {(row_sums==1).sum()} / {len(submission)}')
    print(f'Invalid rows     : {(row_sums!=1).sum()}')
    print()
    
    print('Predicted class distribution in submission:')
    for cls in CLASSES:
        cnt = submission[cls].sum()
        pct = cnt / len(submission) * 100
        print(f'  {cls:<35}: {cnt:5d} ({pct:.1f}%)')

In [ ]:
# ============================================================
# BONUS: GRAD-CAM VISUALIZATION (understand what model sees)
# ============================================================
if TORCH_AVAILABLE:
    def get_gradcam(model, img_tensor, target_class, layer_name='backbone'):
        """
        Compute Grad-CAM heatmap for a single image.
        Uses hooks to capture gradients and activations from the last conv layer.
        """
        model.eval()
        
        # Get last conv layer
        # For EfficientNet in timm, the last block before pooling
        target_layer = model.backbone.blocks[-1]
        
        activations = {}
        gradients   = {}
        
        def save_activation(name):
            def hook(module, input, output):
                activations[name] = output.detach()
            return hook
        
        def save_gradient(name):
            def hook(module, grad_input, grad_output):
                gradients[name] = grad_output[0].detach()
            return hook
        
        h1 = target_layer.register_forward_hook(save_activation('act'))
        h2 = target_layer.register_full_backward_hook(save_gradient('grad'))
        
        img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
        img_tensor.requires_grad_(True)
        
        logits = model(img_tensor)
        model.zero_grad()
        logits[0, target_class].backward()
        
        h1.remove()
        h2.remove()
        
        # Pool gradients over spatial dimensions
        weights = gradients['grad'].mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam = (weights * activations['act']).sum(dim=1).squeeze(0)   # (H, W)
        cam = F.relu(cam)
        cam = cam.cpu().numpy()
        
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        return cam
    
    
    # Visualize for a few validation samples
    import matplotlib.cm as cm
    
    fig, axes = plt.subplots(4, 3, figsize=(15, 20))
    viz_classes_gradcam = ['Atelectasis', 'Cardiomegaly', 'Pneumothorax', 'Effusion']
    
    for row_idx, cls in enumerate(viz_classes_gradcam):
        c_idx = class_to_idx[cls]
        
        cls_rows = df_val[df_val['label'] == c_idx]
        if len(cls_rows) == 0:
            continue
        
        sample_row = cls_rows.iloc[0]
        img_path = os.path.join(IMAGE_DIR, sample_row['id'])
        orig_img = Image.open(img_path).convert('RGB')
        img_tensor = val_transform(orig_img)
        
        try:
            heatmap = get_gradcam(model, img_tensor, c_idx)
        except Exception as e:
            print(f'Grad-CAM failed for {cls}: {e}')
            continue
        
        # Resize heatmap to image size
        from PIL import Image as PILImage
        _resample = getattr(PILImage, 'Resampling', PILImage).BILINEAR
        heatmap_resized = np.array(
            PILImage.fromarray((heatmap * 255).astype(np.uint8))
            .resize((CFG['val_img_size'], CFG['val_img_size']), _resample)
        ) / 255.0
        
        # Display
        axes[row_idx, 0].imshow(orig_img, cmap='gray')
        axes[row_idx, 0].set_title(f'{cls}\nOriginal', fontsize=10)
        axes[row_idx, 0].axis('off')
        
        axes[row_idx, 1].imshow(heatmap_resized, cmap='jet')
        axes[row_idx, 1].set_title('Grad-CAM Heatmap', fontsize=10)
        axes[row_idx, 1].axis('off')
        
        # Overlay
        orig_gray = np.array(orig_img.convert('L')) / 255.0
        overlay = 0.6 * orig_gray[:, :, np.newaxis] * np.ones((1,1,3))
        jet_colors = cm.jet(heatmap_resized)[..., :3]
        overlay += 0.4 * jet_colors
        overlay = overlay.clip(0, 1)
        
        axes[row_idx, 2].imshow(overlay)
        axes[row_idx, 2].set_title('Overlay', fontsize=10)
        axes[row_idx, 2].axis('off')
    
    plt.suptitle('Grad-CAM: Model Attention Maps', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'gradcam_visualization.png'), dpi=120)
    plt.show()

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print('=' * 70)
print('COMPETITION NOTEBOOK SUMMARY')
print('=' * 70)
print()
print('Dataset:')
print(f'  Train: 51,043 images | Test: 17,015 images')
print(f'  Image size: 384×384 RGB | Classes: 20 (single-label)')
print(f'  Imbalance ratio: 6815x (No Finding 66.8% vs Pneumomediastinum 0.01%)')
print()
print('Scoring:')
print(f'  Score_c = (TP - FP - 5×FN) / N_c  →  macro-average')
print(f'  FN penalty 5× FP → minimize missed diagnoses!')
print()
print('Optimal Decision Rule (Bayes-optimal for this scoring):')
print(f'  c* = argmax_c [(7×P(c|x) - 1) / N_c]')
print(f'  → Rare class needs only ~14.3% probability to override common class at 50%')
print()
print('Key Design Choices:')
print(f'  Model    : EfficientNet-B4 Noisy Student (tf_efficientnet_b4_ns)')
print(f'  Loss     : {CFG["loss_fn"]} + class weights (inverse frequency)')
print(f'  Sampling : Weighted random sampler for balanced mini-batches')
print(f'  Aug      : RandomFlip + Rotation + Affine + ColorJitter + RandomErasing + Mixup')
print(f'  Inference: TTA ({CFG["tta_n"]} augmentations) + optimal decision rule')
if TORCH_AVAILABLE and len(history.get('val_score', [])) > 0:
    print()
    print(f'Results:')
    print(f'  Best Val Score: {best_score:.4f}')
print('=' * 70)